In [1]:
import pandas as pd

# Load the CSV file into a Pandas DataFrame
df = pd.read_csv('dataset.csv')

# Print the shape to confirm we have all 10,000+ rows
print(f"Dataset Shape: {df.shape}\n")

# Let's look at the first row's transcript to see the dialogue structure
print("--- Sample Transcript Preview ---")
print(df['Transcript'].iloc[0][:500]) # Printing just the first 500 characters to get a sense of the dialogue format

Dataset Shape: (10174, 8)

--- Sample Transcript Preview ---
Interviewer: Good morning, Jason. It's great to meet you. Welcome to the interview for the E-commerce Specialist role at our company.

Jason Jones: Good morning. Thank you for having me. It's a pleasure to be here.

Interviewer: Before we begin, I want to let you know that this interview will cover various aspects of the role, including customer service, product listing, and other relevant topics. Can you start by telling me a little bit about your background and how you think your skills and ex


In [3]:
import re

def extract_candidate_answers(row):
    """
    Takes a row of the dataframe, finds the candidate's name, and uses Regex 
    to extract only what they said from the transcript.
    """
    transcript = str(row['Transcript'])
    candidate_name = str(row['Name'])
    
    # We use regex to find: [Candidate Name]: [Everything they say until "Interviewer:" or the end of the text]
    # re.escape ensures if a name has weird punctuation, it doesn't break our regex
    pattern = rf"{re.escape(candidate_name)}:\s*(.*?)(?=Interviewer:|$)"
    
    # re.DOTALL allows us to capture multi-line answers
    # re.IGNORECASE ensures we don't fail if the casing is slightly off
    matches = re.findall(pattern, transcript, flags=re.IGNORECASE | re.DOTALL)
    
    # Join all their individual answers into one continuous string
    cleaned_text = " ".join(matches).strip()
    
    # Fallback: If our regex fails (maybe the transcript format was slightly different), 
    # we return the original so we don't lose the data, but we flag it.
    if not cleaned_text:
        return transcript 
        
    return cleaned_text

# Apply this function to every row in our dataset and create a new column
print("Cleaning transcripts... this might take a few seconds.")
df['Cleaned_Transcript'] = df.apply(extract_candidate_answers, axis=1)

# Let's compare the original vs. the cleaned version for our friend Jason
print("\n--- ORIGINAL TRANSCRIPT (First 300 chars) ---")
print(df['Transcript'].iloc[0][:300])

print("\n--- CLEANED TRANSCRIPT (First 300 chars) ---")
print(df['Cleaned_Transcript'].iloc[0][:300])

Cleaning transcripts... this might take a few seconds.

--- ORIGINAL TRANSCRIPT (First 300 chars) ---
Interviewer: Good morning, Jason. It's great to meet you. Welcome to the interview for the E-commerce Specialist role at our company.

Jason Jones: Good morning. Thank you for having me. It's a pleasure to be here.

Interviewer: Before we begin, I want to let you know that this interview will cover 

--- CLEANED TRANSCRIPT (First 300 chars) ---
Good morning. Thank you for having me. It's a pleasure to be here.

 Uh, sure. I have about 3 years of experience in e-commerce, working with online marketplaces like Amazon and eBay. My most recent role was as a customer service representative for an online retailer, where I handled customer inquir


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# 1. Convert the Target Variable into binary numbers (1 for select, 0 for reject)
# We will create a new column called 'Target'
df['Target'] = df['decision'].map({'select': 1, 'reject': 0})

# 2. Initialize the TF-IDF Vectorizer
# We limit to the top 5,000 most important words to keep the matrix manageable 
# We also remove standard English 'stop words' (and, the, is, etc.)
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')

# 3. Fit and Transform the Cleaned Transcripts into a numerical matrix (X)
print("Vectorizing text... converting words to math.")
X = tfidf.fit_transform(df['Cleaned_Transcript'])

# 4. Define our labels (y)
y = df['Target']

print(f"Feature Matrix (X) shape: {X.shape}")
print(f"Target Vector (y) shape: {y.shape}")

Vectorizing text... converting words to math.
Feature Matrix (X) shape: (10174, 5000)
Target Vector (y) shape: (10174,)


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Split the Data
# We use 80% of the data to train the model, and hold back 20% to test it.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training on {X_train.shape[0]} interviews...")
print(f"Testing on {X_test.shape[0]} hidden interviews...\n")

# 2. Initialize the Model
# Logistic Regression is highly effective and interpretable for text classification
model = LogisticRegression(max_iter=1000)

# 3. Train the Model (This is where the actual 'learning' happens)
print("Training the model... this might take a moment.")
model.fit(X_train, y_train)

# 4. Make Predictions on the hidden test set
y_pred = model.predict(X_test)

# 5. Evaluate the Results (The Academic Proof)
print("\n--- MODEL EVALUATION ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")

print("Classification Report (Precision, Recall, F1-Score):")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Training on 8139 interviews...
Testing on 2035 hidden interviews...

Training the model... this might take a moment.

--- MODEL EVALUATION ---
Accuracy: 61.92%

Classification Report (Precision, Recall, F1-Score):
              precision    recall  f1-score   support

           0       0.64      0.58      0.61      1034
           1       0.60      0.66      0.63      1001

    accuracy                           0.62      2035
   macro avg       0.62      0.62      0.62      2035
weighted avg       0.62      0.62      0.62      2035

Confusion Matrix:
[[595 439]
 [336 665]]


In [6]:
import joblib

# Save the trained Logistic Regression model
joblib.dump(model, 'clarity_classifier.pkl')

# Save the TF-IDF Vectorizer (so the app knows how to convert new text into the same math)
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')

print("Models successfully saved to disk as .pkl files!")

Models successfully saved to disk as .pkl files!
